# Config

In [1]:
import os

# Ruta a la que quieres mover el path
nueva_ruta = "/tmp/Repository/VRID_language_proyect/BERT"

# Cambiar el directorio actual
os.chdir(nueva_ruta)

# Confirmar que cambió
print("Directorio actual:", os.getcwd())

Directorio actual: /tmp/Repository/VRID_language_proyect/BERT


In [2]:
import pandas as pd
import os
from preprocess.preprocess import clean_text, check_deleted_expressions
from preprocess.translate import translator, gen_text_for_embedding, final_clean, detect_language
import time
import json
import numpy as np

# 4) TF ID feature extractor

## Train

In [43]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

#Prueba con solo textos traducidos
#df = df[df["Español"]==False]

In [ ]:
from sklearn.preprocessing import LabelEncoder
from models.TIFD import gen_TFID_vectors
import numpy as np
from utils.dataset import gen_dataset_select_cols

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "keywords_trad", "Depto_Persona_trad"]

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)
#Creacion de vectores TFID
X_train, X_test = gen_TFID_vectors(X_train, X_test)
print(X_train.shape, X_test.shape)

(771, 4367) (193, 4367)


In [89]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring='f1_macro'
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [1.0, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', 'lbfgs'] before, using random point [10, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [0.1, 'l2', '

RandomForestClassifier
Compute sw
XGBClassifier
Compute sw
SVC
Compute sw

🔍 Validación:
LogisticRegression: {'mean_test_score': 0.69, 'std_test_score': 0.03}
RandomForestClassifier: {'mean_test_score': 0.72, 'std_test_score': 0.01}
XGBClassifier: {'mean_test_score': 0.73, 'std_test_score': 0.02}
SVC: {'mean_test_score': 0.68, 'std_test_score': 0.01}


In [90]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.7046632124352331, 'precision': 0.75, 'recall': 0.7297297297297297, 'f1_macro': 0.6992043310638996, 'cm': array([[55, 27],
       [30, 81]]), 'f1_es': 0.7102667153818049, 'f1_en': 0.6885780885780887, 'cm_es': array([[22, 18],
       [15, 60]]), 'cm_en': array([[33,  9],
       [15, 21]])}
RandomForestClassifier
{'accuracy': 0.689119170984456, 'precision': 0.7256637168141593, 'recall': 0.7387387387387387, 'f1_macro': 0.6808862433862434, 'cm': array([[51, 31],
       [29, 82]]), 'f1_es': 0.7133247722130496, 'f1_en': 0.6366744366744367, 'cm_es': array([[20, 20],
       [12, 63]]), 'cm_en': array([[31, 11],
       [17, 19]])}
XGBClassifier
{'accuracy': 0.694300518134715, 'precision': 0.7549019607843137, 'recall': 0.6936936936936937, 'f1_macro': 0.6909821162039675, 'cm': array([[57, 25],
       [34, 77]]), 'f1_es': 0.7217391304347827, 'f1_en': 0.6458702587734846, 'cm_es': array([[24, 16],
       [16, 59]]), 'cm_en': array([[33,  9],
       [18, 18]])}
SVC
{'

## Save

In [91]:
exp_info = {
    'exp_name': "Bayesiansearchcv_TFID_department",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "cols": cols
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

Current tracking uri: http://mlflow-server:5000
📝 Registrando modelo en MLflow: LogisticRegression


2025/09/03 19:23:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/7/runs/75311954004d42489d9b7597a7cde732
🧪 View experiment at: http://mlflow-server:5000/#/experiments/7
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/09/03 19:23:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/7/runs/19ee0885458646bb9bcbf8191ad6c961
🧪 View experiment at: http://mlflow-server:5000/#/experiments/7
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/03 19:23:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/7/runs/10d123cf0d274f76b8d648b3722a456e
🧪 View experiment at: http://mlflow-server:5000/#/experiments/7
📝 Registrando modelo en MLflow: SVC


2025/09/03 19:23:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run SVC at: http://mlflow-server:5000/#/experiments/7/runs/f0c1ede2898f4e729141ed38f715de55
🧪 View experiment at: http://mlflow-server:5000/#/experiments/7


# 5) SPECTER model

In [11]:
import numpy as np
import pandas as pd
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset
from preprocess.translate import gen_text_for_embedding, final_clean

def gen_dataset_select_cols(codes_vrid, df, cols, element_names=None):
    #Selección unicamente de elementos de df que se encuentren en codes_vrid
    df = df[df["Código VRID"].isin(codes_vrid)].copy()

    #Creación de index en función de orden de los datos
    df['idx'] = np.arange(0, df.shape[0])

    #Generación de datasets
    for col in cols:
        df[col] = df[col].apply(final_clean)

    df=gen_text_for_embedding(df, cols, element_names=element_names)
    X = df["text_for_embedding_translated"].to_list()
    y = df["Interdisciplinario"].to_list()
    return X, y, df

def decoder_vrid(fold_codes, df_decode):
    """
    Decodifica fold_codes usando df_decode.
    fold_codes : lista o array con índices (ej. [0, 2, 5])
    df_decode  : DataFrame con columnas ['idx', 'Código VRID']

    Devuelve un numpy.array con los códigos VRID correspondientes.
    """
    # Crear un diccionario {Código VRID: código}
    mapping = df_decode.set_index("Código VRID")["idx"].to_dict()

    # Mapear los fold_codes a códigos (ignora los que no existan en mapping)
    decoded = [mapping[c] for c in fold_codes if c in mapping]

    return np.array(decoded)

In [6]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [20]:
df.columns


Index(['Código VRID', 'Interdisciplinario', 'Transdisciplinario', 'Título',
       'Keywords', 'Resumen', 'Facultad del Proyecto', 'Depto Persona',
       'Titulo_trad', 'Resumen_trad', 'keywords_trad',
       'Facultad_del_Proyecto_trad', 'Depto_Persona_trad', 'Español',
       'text_for_embedding_translated'],
      dtype='object')

In [22]:
#del gen_dataset
#from utils.dataset import gen_dataset_select_cols
from models.specter import embed_texts
import numpy as np
from sklearn.preprocessing import LabelEncoder

#Columnas a seleccionar para clasificación
cols = ["Titulo_trad", "Depto_Persona_trad", "keywords_trad", "Resumen_trad"]
element_names=["Title:", "Department:", "keywords:", "abstract:"]

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset_select_cols(codes_test, df, cols = cols, element_names=element_names)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset_select_cols(codes_train, df, cols = cols)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# 2) Calcular embeddings
# Parámetros modelo
BASE_MODEL = "allenai/specter2_base"
#ADAPTER_NAME = "allenai/specter2"
ADAPTER_NAME="allenai/specter2_classification"
X_train = embed_texts(X_train, BASE_MODEL, ADAPTER_NAME)
X_test = embed_texts(X_test, BASE_MODEL, ADAPTER_NAME)

print(X_train.shape, X_test.shape)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.


Modelo SPECTER2 cargado correctamente.
(771, 768) (193, 768)


In [ ]:
from pipelines.ML_pipeline_skp import get_est_params_dict, run_bayesian_pipeline, select_best_model, eval_model, mlflow_ckeckpoint
from utils.dataset import CvCustom
from collections import Counter

# 2. Elegir modelos a probar
model_keys = [
    'LogisticRegression',
    #'DecisionTreeClassifier',
    'RandomForestClassifier',
    #'GradientBoostingClassifier',
    'XGBClassifier',
    #'MLPClassifier',
    #'SVC',
    #'SGDClassifier'
]

# 3. Obtener el diccionario de modelos y parámetros
est_params_dict = get_est_params_dict(model_keys)

print("📊 train:", Counter(y_train))
print("📊 test:", Counter(y_test))
# 4. Ejecutar entrenamiento, validación y test con tus funciones
n_iter=20
sample_weight_On=True
scoring="f1_macro"
results_val, models_dicc = run_bayesian_pipeline(est_params_dict, X_train, y_train, scoring=scoring, cv_function=CvCustom(df_decode), 
                                                 n_iter=n_iter, sample_weight_On = sample_weight_On)

best_model = select_best_model(results_val, models_dicc)

# 5. Mostrar resultados
print("\n🔍 Validación:")
for model, metrics in results_val.items():
    print(f"{model}: {metrics}")

📊 train: Counter({1: 444, 0: 327})
📊 test: Counter({1: 111, 0: 82})
(771,)
LogisticRegression
Compute sw


/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [1.0, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [1.0, 'l2', 'lbfgs'] before, using random point [0.1, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbfgs'] before, using random point [10, 'l2', 'lbfgs']
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/skopt/optimizer/optimizer.py:517: UserWarning: The objective has been evaluated at point [10, 'l2', 'lbf

RandomForestClassifier
Compute sw


In [ ]:
# Métricas por idioma
lang_es = df_test["Español"]
for name, model in models_dicc.items():
    print(name)
    results=eval_model(model, X_test, y_test, lang_es)
    print(results)

LogisticRegression
{'accuracy': 0.6424870466321243, 'precision': 0.647887323943662, 'recall': 0.8288288288288288, 'f1_macro': 0.6042378673957622, 'cm': array([[32, 50],
       [19, 92]]), 'f1_es': 0.6795950820307955, 'f1_en': 0.535419096394706, 'cm_es': array([[13, 27],
       [ 6, 69]]), 'cm_en': array([[19, 23],
       [13, 23]])}
RandomForestClassifier
{'accuracy': 0.6632124352331606, 'precision': 0.6575342465753424, 'recall': 0.8648648648648649, 'f1_macro': 0.6216028715350044, 'cm': array([[32, 50],
       [15, 96]]), 'f1_es': 0.6550102474015518, 'f1_en': 0.6007977207977208, 'cm_es': array([[10, 30],
       [ 4, 71]]), 'cm_en': array([[22, 20],
       [11, 25]])}
XGBClassifier
{'accuracy': 0.6373056994818653, 'precision': 0.6518518518518519, 'recall': 0.7927927927927928, 'f1_macro': 0.6077235772357724, 'cm': array([[35, 47],
       [23, 88]]), 'f1_es': 0.6486230277322537, 'f1_en': 0.5870391967952944, 'cm_es': array([[14, 26],
       [12, 63]]), 'cm_en': array([[21, 21],
       [11,

## Save

In [ ]:
exp_info = {
    'exp_name': "Bayesiansearchcv_specter_department",
    #'artifact_path': "file:///tmp/mlflow_experiments/mlruns", 
    #'tracking_path': "sqlite:////tmp/mlflow_experiments/mlflow.db",
}

extra_parms = {
    "n_iter": n_iter,
    "sample_weight_On": sample_weight_On,
    "scoring": scoring,
    "cols": cols
}

mlflow_ckeckpoint(exp_info, results_val, models_dicc, extra_parms, X_test, y_test, lang_es)

Current tracking uri: http://mlflow-server:5000


2025/09/03 20:50:21 INFO mlflow.tracking.fluent: Experiment with name 'Bayesiansearchcv_specter_department' does not exist. Creating a new experiment.


📝 Registrando modelo en MLflow: LogisticRegression


2025/09/03 20:50:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run LogisticRegression at: http://mlflow-server:5000/#/experiments/8/runs/cbe80d3ece554f76b9cb97e1fbd1a5fb
🧪 View experiment at: http://mlflow-server:5000/#/experiments/8
📝 Registrando modelo en MLflow: RandomForestClassifier


2025/09/03 20:50:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run RandomForestClassifier at: http://mlflow-server:5000/#/experiments/8/runs/13ec7dd538a04878a8f0990d81076216
🧪 View experiment at: http://mlflow-server:5000/#/experiments/8
📝 Registrando modelo en MLflow: XGBClassifier


2025/09/03 20:50:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://mlflow-server:5000/#/experiments/8/runs/5513b4333bf74610bd199046143f8262
🧪 View experiment at: http://mlflow-server:5000/#/experiments/8


# 6) all-roberta-large-v1

In [ ]:
#Lectura de index de separacion de conjuntos train/test
path = "/tmp/data"
filepath=os.path.join(path, "train_test_ids_3folds.json")
with open(filepath, "r", encoding="utf-8") as f:
    dataset_index = json.load(f)

#Lectura de data
path = "/tmp/data"
filepath=os.path.join(path, "data_translated_concat.csv")
df = pd.read_csv(filepath)

In [ ]:
#del gen_dataset
from utils.dataset import gen_dataset
from models.specter import embed_texts
from sklearn.preprocessing import LabelEncoder

#Lectura de codigos 
codes_test = dataset_index["Test"]
X_test, y_test, df_test= gen_dataset(codes_test, df)

#Lectura de codigos 
codes_train = dataset_index["kfolds"]
codes_train = np.array([i for fold in codes_train for i in fold])
X_train, y_train, df_train = gen_dataset(codes_train, df)
df_decode = df_train[["idx", "Código VRID"]]

# Codificación de labels
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)